# Assignment 1 — training pipeline

You're using a notebook for this stage. If you'd rather use a script,
write `train.py` instead and delete this file — `main.py` refuses to run
if it finds both `train.py` and `train.ipynb`, so exactly one of them must
exist.

Replace this cell and everything below it with your own code. When run via
`uv run python main.py train`, this notebook must, start to finish, with
no manual steps in between:

   # Online Shoppers: who gets the discount pop-up?
   Logistic regression trained three ways on a month-based split. See REPORT.md for the framing.
  

How you structure the code beyond that — cell layout, function
boundaries, naming — is your call to make and be able to defend.

## 1. Load, deduplicate, split by month

In [1]:
import numpy as np, pandas as pd, joblib
from pathlib import Path
ART = Path("artifacts"); ART.mkdir(exist_ok=True)

df = pd.read_csv("data/online_shoppers_intention.csv")
n0 = len(df); df = df.drop_duplicates().reset_index(drop=True)
print(f"Dropped {n0 - len(df)} duplicates -> {len(df)} rows")

TRAIN_M = ["Feb", "Mar", "May", "June", "Jul", "Aug", "Sep"]
VAL_M = ["Oct", "Nov"]
df["split"] = np.select([df.Month.isin(TRAIN_M), df.Month.isin(VAL_M)], ["train", "val"], "test")
print(df.groupby("split")["Revenue"].agg(["count", "mean"]))

Dropped 125 duplicates -> 12205 rows
       count      mean
split                 
test    1706  0.126612
train   6968  0.117250
val     3531  0.247805


## 2. Feature engineering

In [2]:
def engineer(d: pd.DataFrame) -> pd.DataFrame:
    d = d.copy()
    eps = 1e-6
    d["total_pages"] = d.Administrative + d.Informational + d.ProductRelated
    d["total_duration"] = d.Administrative_Duration + d.Informational_Duration + d.ProductRelated_Duration
    d["product_time_share"] = d.ProductRelated_Duration / (d.total_duration + eps)   # focus on products
    d["sec_per_product_page"] = d.ProductRelated_Duration / (d.ProductRelated + eps) # engagement depth
    d["single_page"] = (d.total_pages <= 1).astype(int)                              # instant bounce flag
    d["has_pagevalue"] = (d.PageValues > 0).astype(int)
    d["is_new_visitor"] = (d.VisitorType == "New_Visitor").astype(int)
    d["weekend"] = d.Weekend.astype(int)
    skewed = ["Administrative", "Administrative_Duration", "Informational", "Informational_Duration",
              "ProductRelated", "ProductRelated_Duration", "total_pages", "total_duration",
              "sec_per_product_page", "PageValues"]
    for c in skewed:
        d["log_" + c] = np.log1p(d[c])   # long right tails -> log1p
    return d

df = engineer(df)

NUM = ["log_Administrative", "log_Administrative_Duration", "log_Informational",
       "log_Informational_Duration", "log_ProductRelated", "log_ProductRelated_Duration",
       "log_total_pages", "log_total_duration", "log_sec_per_product_page",
       "product_time_share", "BounceRates", "ExitRates", "SpecialDay",
       "single_page", "is_new_visitor", "weekend"]
PV = ["log_PageValues", "has_pagevalue"]
CAT = ["OperatingSystems", "Browser", "Region", "TrafficType"]   # ID codes, not quantities

## 3. Preprocessing (fitted on train only)

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

def make_prep(num_cols):
    return ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=50,
                              sparse_output=False), CAT),   # rare codes grouped together
    ])

tr, va, te = (df[df.split == s] for s in ["train", "val", "test"])
y = {s: d["Revenue"].astype(int).to_numpy() for s, d in zip(["train", "val", "test"], [tr, va, te])}

data = {}
for name, cols in {"full": NUM + PV, "deployable": NUM}.items():
    prep = make_prep(cols).fit(tr)                       # fit on TRAIN only
    data[name] = {s: prep.transform(d).astype(np.float32)
                  for s, d in zip(["train", "val", "test"], [tr, va, te])}
    joblib.dump(prep, ART / f"prep_{name}.joblib")
    print(name, data[name]["train"].shape)

full (6968, 52)
deployable (6968, 50)


## 4. Business cost and threshold tuning

In [4]:
import json, torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss
torch.manual_seed(0)

C_FP, C_FN = 2.0, 10.0      # € per false positive / false negative (see REPORT)
C_REG = 1.0                 # sklearn L2 strength; the SAME penalty is used in both torch versions
EPOCHS, LR = 10000, 1.0     # fewer epochs left GD unconverged (collinear features), see REPORT

def business_cost(y_true, p, t):
    pred = p >= t
    return C_FP * np.sum(pred & (y_true == 0)) + C_FN * np.sum(~pred & (y_true == 1))

def best_threshold(y_true, p):
    grid = np.round(np.arange(0.01, 1.0, 0.01), 2)
    return float(grid[int(np.argmin([business_cost(y_true, p, t) for t in grid]))])

## 5. Three implementations of logistic regression

In [5]:
def train_sklearn(X, y_):
    return LogisticRegression(C=C_REG, max_iter=5000).fit(X, y_)

def train_manual(X, y_):
    Xt = torch.tensor(X); yt = torch.tensor(y_, dtype=torch.float32)
    n, d = Xt.shape; lam = 1.0 / (C_REG * n)          # same objective as sklearn's L2
    w = torch.zeros(d, requires_grad=True); b = torch.zeros(1, requires_grad=True)
    hist = []
    for _ in range(EPOCHS):
        p = torch.sigmoid(Xt @ w + b)
        bce = -(yt * torch.log(p + 1e-7) + (1 - yt) * torch.log(1 - p + 1e-7)).mean()
        loss = bce + 0.5 * lam * (w ** 2).sum()        # L2 on weights, not on bias
        loss.backward()
        with torch.no_grad():
            w -= LR * w.grad
            b -= LR * b.grad
        w.grad.zero_(); b.grad.zero_()
        hist.append(bce.item())
    return {"w": w.detach(), "b": b.detach()}, hist

class LogReg(torch.nn.Module):
    def __init__(self, d):
        super().__init__()
        self.linear = torch.nn.Linear(d, 1)
    def forward(self, x):
        return self.linear(x).squeeze(-1)              # logits

def train_standard(X, y_):
    Xt = torch.tensor(X); yt = torch.tensor(y_, dtype=torch.float32)
    n, d = Xt.shape; lam = 1.0 / (C_REG * n)
    model = LogReg(d)
    torch.nn.init.zeros_(model.linear.weight); torch.nn.init.zeros_(model.linear.bias)
    opt = torch.optim.SGD([{"params": [model.linear.weight], "weight_decay": lam},
                           {"params": [model.linear.bias], "weight_decay": 0.0}], lr=LR)
    loss_fn = torch.nn.BCEWithLogitsLoss()
    hist = []
    for _ in range(EPOCHS):
        opt.zero_grad()
        loss = loss_fn(model(Xt), yt)
        loss.backward(); opt.step()
        hist.append(loss.item())
    return model, hist

def predict(method, m, X):
    if method == "sklearn":
        return m.predict_proba(X)[:, 1]
    Xt = torch.tensor(X)
    with torch.no_grad():
        if method == "manual":
            return torch.sigmoid(Xt @ m["w"] + m["b"]).numpy()
        return torch.sigmoid(m(Xt)).numpy()

## 6. Train, tune threshold on validation, evaluate on test

In [6]:
rows, histories, models, test_preds = [], {}, {}, {}
base_rate = y["train"].mean()
pb_val, pb_te = np.full(len(y["val"]), base_rate), np.full(len(y["test"]), base_rate)
tb = best_threshold(y["val"], pb_val)
rows.append(dict(features="-", method="baseline", threshold=tb, test_auc=0.5,
                 test_logloss=log_loss(y["test"], pb_te), test_cost_eur=business_cost(y["test"], pb_te, tb)))

for fs in ["full", "deployable"]:
    X = data[fs]
    sk = train_sklearn(X["train"], y["train"])
    man, h_man = train_manual(X["train"], y["train"])
    std, h_std = train_standard(X["train"], y["train"])
    histories[fs] = {"manual": h_man, "standard": h_std}
    for method, m in [("sklearn", sk), ("manual", man), ("standard", std)]:
        models[(fs, method)] = m
        pv, pt = predict(method, m, X["val"]), predict(method, m, X["test"])
        t = best_threshold(y["val"], pv)
        test_preds[f"{fs}_{method}"] = pt
        rows.append(dict(features=fs, method=method, threshold=t,
                         test_auc=roc_auc_score(y["test"], pt), test_logloss=log_loss(y["test"], pt),
                         test_cost_eur=business_cost(y["test"], pt, t)))

results = pd.DataFrame(rows)
print(results.round(4).to_string(index=False))
print("Policy 'show to nobody': €", C_FN * y["test"].sum(), "| 'show to everyone': €", C_FP * (y["test"] == 0).sum())

  features   method  threshold  test_auc  test_logloss  test_cost_eur
         - baseline       0.01    0.5000        0.3803         2980.0
      full  sklearn       0.05    0.9000        0.2234          762.0
      full   manual       0.05    0.9000        0.2230          764.0
      full standard       0.05    0.9000        0.2230          764.0
deployable  sklearn       0.10    0.7485        0.3366         1916.0
deployable   manual       0.10    0.7481        0.3367         1920.0
deployable standard       0.10    0.7481        0.3367         1920.0
Policy 'show to nobody': € 2160.0 | 'show to everyone': € 2980.0


## 7. Do the three implementations agree?

In [7]:
for fs in ["full", "deployable"]:
    w = {"sklearn": models[(fs, "sklearn")].coef_.ravel(),
         "manual": models[(fs, "manual")]["w"].numpy(),
         "standard": models[(fs, "standard")].linear.weight.detach().numpy().ravel()}
    p = {k: test_preds[f"{fs}_{k}"] for k in w}
    print(fs)
    for a, b in [("sklearn", "manual"), ("sklearn", "standard"), ("manual", "standard")]:
        print(f"  {a:>8} vs {b:<8} max|coef diff| {np.abs(w[a]-w[b]).max():.4f}"
              f"  max|prob diff| {np.abs(p[a]-p[b]).max():.4f}")

full
   sklearn vs manual   max|coef diff| 0.1556  max|prob diff| 0.0316
   sklearn vs standard max|coef diff| 0.1556  max|prob diff| 0.0316
    manual vs standard max|coef diff| 0.0000  max|prob diff| 0.0000
deployable
   sklearn vs manual   max|coef diff| 0.2214  max|prob diff| 0.0129
   sklearn vs standard max|coef diff| 0.2214  max|prob diff| 0.0129
    manual vs standard max|coef diff| 0.0000  max|prob diff| 0.0000


## 8. Save artifacts for the dashboard

In [9]:
for fs in ["full", "deployable"]:
    joblib.dump(models[(fs, "sklearn")], ART / f"sklearn_{fs}.joblib")
    torch.save(models[(fs, "manual")], ART / f"manual_{fs}.pt")
    torch.save(models[(fs, "standard")].state_dict(), ART / f"standard_{fs}.pt")
    np.save(ART / f"X_test_{fs}.npy", data[fs]["test"])
np.save(ART / "y_test.npy", y["test"])
te.to_csv(ART / "test_raw.csv", index=False)
tr.to_csv(ART / "train_raw.csv", index=False)
va.to_csv(ART / "val_raw.csv", index=False)
results.to_csv(ART / "results.csv", index=False)
json.dump({"C_FP": C_FP, "C_FN": C_FN, "base_rate": float(base_rate), "histories": histories},
          open(ART / "meta.json", "w"))
print("Saved:", sorted(p.name for p in ART.iterdir()))

Saved: ['X_test_deployable.npy', 'X_test_full.npy', 'manual_deployable.pt', 'manual_full.pt', 'meta.json', 'prep_deployable.joblib', 'prep_full.joblib', 'results.csv', 'sklearn_deployable.joblib', 'sklearn_full.joblib', 'standard_deployable.pt', 'standard_full.pt', 'test_raw.csv', 'train_raw.csv', 'val_raw.csv', 'y_test.npy']
